# Random Forest Classification

Ky notebook trajnon dhe vlereson nje model Random Forest per klasifikimin e cmimeve ne klasa.

## 1. Importimi i librarive dhe ngarkimi i dataset-it

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
import seaborn as sns
from sklearn.tree import plot_tree

RANDOM_STATE = 42

# Gjejme folderin kryesor te projektit, pavaresisht nga ku hapet notebook-u.
project_root = Path.cwd()
while not (project_root / "data" / "processed").exists() and project_root != project_root.parent:
    project_root = project_root.parent

data_dir = project_root / "data" / "processed"
# Per Random Forest perdorim dataset-et unscaled, sepse modelet me peme nuk kane nevoje per scaling.
train_path = data_dir / "train_unscaled_dataset.csv"
val_path = data_dir / "val_unscaled_dataset.csv"
test_path = data_dir / "test_unscaled_dataset.csv"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print("Train dataset:", train_path)
print("Validation dataset:", val_path)
print("Test dataset:", test_path)

## 2. Kontroll i shkurter i te dhenave

In [ ]:
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())
print("\nMissing values:")
print("Train:", train_df.isna().sum().sum())
print("Validation:", val_df.isna().sum().sum())
print("Test:", test_df.isna().sum().sum())

print("\nData types:")
print(train_df.dtypes.value_counts())

## 3. Ndarja e features dhe target-it

In [ ]:
target_col = "price_class"

feature_cols = [col for col in train_df.columns if col != target_col]

X_train = train_df[feature_cols].copy()
y_train = train_df[target_col]

X_val = val_df[feature_cols].copy()
y_val = val_df[target_col]

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col]

## 4. Defino funksionin evaluate_model

In [ ]:
def evaluate_model(model, X, y, model_name):
    y_pred = model.predict(X)

    results = {
        "Model": model_name,
        "Accuracy": accuracy_score(y, y_pred),
        "Precision Macro": precision_score(y, y_pred, average="macro", zero_division=0),
        "Recall Macro": recall_score(y, y_pred, average="macro", zero_division=0),
        "F1 Macro": f1_score(y, y_pred, average="macro", zero_division=0),
    }

    print(model_name)
    print("Accuracy:", round(results["Accuracy"], 4))
    print("Precision Macro:", round(results["Precision Macro"], 4))
    print("Recall Macro:", round(results["Recall Macro"], 4))
    print("F1 Macro:", round(results["F1 Macro"], 4))

    print("\nClassification report:")
    print(classification_report(y, y_pred, zero_division=0))

    return results, y_pred

## 5. Krahasimi i konfigurimeve Random Forest ne validation

In [ ]:
rf_models = {
    "RF - Gini": RandomForestClassifier(
        n_estimators=100,
        criterion="gini",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "RF - Entropy": RandomForestClassifier(
        n_estimators=100,
        criterion="entropy",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "RF - Max Depth 10": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

validation_results = []

for name, model in rf_models.items():
    model.fit(X_train, y_train)
    results, _ = evaluate_model(model, X_val, y_val, name)
    validation_results.append(results)

validation_results_df = pd.DataFrame(validation_results).sort_values("F1 Macro", ascending=False)
display(validation_results_df)

best_model_name = validation_results_df.iloc[0]["Model"]
best_rf = rf_models[best_model_name]

print("Best model based on validation F1 Macro:", best_model_name)

## 6. Vizualizimi i pemëve të vendimit

Random Forest përbëhet nga shumë pemë vendimi. Për interpretim, vizualizohet një pemë përfaqësuese nga secili konfigurim i modelit. Thellësia kufizohet në `max_depth=3` që grafiku të jetë i lexueshëm.

In [ ]:
for name, model in rf_models.items():
    plt.figure(figsize=(22, 10))

    plot_tree(
        model.estimators_[0],
        feature_names=X_train.columns,
        class_names=[str(c) for c in model.classes_],
        filled=True,
        rounded=True,
        max_depth=3,
        fontsize=8
    )

    plt.title(f"Decision Tree Example from {name}")
    plt.tight_layout()
    plt.show()

## 7. Testimi final ne test set

In [ ]:
test_results, test_pred = evaluate_model(
    best_rf,
    X_test,
    y_test,
    f"{best_model_name} - Test"
)

## 8. Confusion Matrix per test

In [ ]:
labels = best_rf.classes_

plt.figure(figsize=(8, 6))
sns.heatmap(
    confusion_matrix(y_test, test_pred, labels=labels),
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)
plt.title(f"Confusion Matrix - {best_model_name} Test")
plt.ylabel("Aktuale")
plt.xlabel("Te parashikuara")
plt.tight_layout()
plt.show()

## 9. Feature Importance

Random Forest mundëson analizën e rëndësisë së features. Kjo tregon cilat karakteristika kanë kontribuar më shumë në ndarjen e klasave të çmimit.

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_rf.feature_importances_
}).sort_values("Importance", ascending=False)

display(feature_importance.head(10))

plt.figure(figsize=(8, 5))
sns.barplot(
    data=feature_importance.head(10),
    x="Importance",
    y="Feature"
)
plt.title(f"Top 10 Feature Importances - {best_model_name}")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()